In [0]:
CREATE OR REPLACE TABLE acme_catalog.gold.agg_sales_by_month AS
SELECT
    dd.year, dd.quarter, dd.month, dd.month_name, 
    dc.division_id, dp.category_id,
    CAST(SUM(fol.revenue) AS DECIMAL(14,2)) as total_revenue,
    CAST(SUM(fol.cost) AS DECIMAL(14,2)) as total_cost,
    CAST(SUM(fol.margin) AS DECIMAL(14,2)) as total_margin,
    -- DODAJTE OVO:
    CAST(SUM(fo.freight) AS DECIMAL(14,2)) as total_freight,
    CAST(SUM(fol.margin) - SUM(fo.freight) AS DECIMAL(14,2)) as net_margin,
    ROUND((SUM(fol.margin) - SUM(fo.freight)) / NULLIF(SUM(fol.revenue), 0) * 100, 2) as net_margin_pct,
    -- Ostalo:
    COUNT(DISTINCT fo.order_id) as order_count,
    COUNT(DISTINCT fo.customer_sk) as distinct_customers,
    ROUND(SUM(fol.revenue) / COUNT(DISTINCT fo.order_id), 2) as avg_order_value
FROM acme_catalog.gold.fact_order_lines fol
JOIN acme_catalog.gold.fact_orders fo ON fol.order_id = fo.order_id
JOIN acme_catalog.gold.dim_date dd ON fo.date_id = dd.date_id
JOIN acme_catalog.gold.dim_customers dc ON fo.customer_sk = dc.customer_sk AND dc.is_current = TRUE
JOIN acme_catalog.gold.dim_products dp ON fol.product_sk = dp.product_sk AND dp.is_current = TRUE
GROUP BY dd.year, dd.quarter, dd.month, dd.month_name, dc.division_id, dp.category_id;

CREATE OR REPLACE TABLE acme_catalog.gold.agg_yoy_comparison AS
SELECT 
  cy.year,
  cy.quarter,
  cy.month,
  cy.month_name,
  cy.division_id,
  cy.category_id,
  -- Current year metrics
  cy.total_revenue as current_revenue,
  cy.total_cost as current_cost,
  cy.total_margin as current_margin,
  cy.order_count as current_order_count,
  cy.distinct_customers as current_customers,
  -- Last year metrics
  ly.total_revenue as last_year_revenue,
  ly.total_cost as last_year_cost,
  ly.total_margin as last_year_margin,
  ly.order_count as last_year_order_count,
  ly.distinct_customers as last_year_customers,
  -- YoY Revenue change
  cy.total_revenue - COALESCE(ly.total_revenue, 0) as yoy_revenue_change,
  ROUND(((cy.total_revenue - COALESCE(ly.total_revenue, 0)) / NULLIF(ly.total_revenue, 0)) * 100, 2) as yoy_revenue_pct,
  -- YoY Margin change
  cy.total_margin - COALESCE(ly.total_margin, 0) as yoy_margin_change,
  ROUND(((cy.total_margin - COALESCE(ly.total_margin, 0)) / NULLIF(ly.total_margin, 0)) * 100, 2) as yoy_margin_pct,
  -- YoY Order count change
  cy.order_count - COALESCE(ly.order_count, 0) as yoy_order_change,
  ROUND(((cy.order_count - COALESCE(ly.order_count, 0)) / NULLIF(ly.order_count, 0)) * 100, 2) as yoy_order_pct,
  -- YoY Customer count change
  cy.distinct_customers - COALESCE(ly.distinct_customers, 0) as yoy_customer_change,
  ROUND(((cy.distinct_customers - COALESCE(ly.distinct_customers, 0)) / NULLIF(ly.distinct_customers, 0)) * 100, 2) as yoy_customer_pct
FROM acme_catalog.gold.agg_sales_by_month cy
LEFT JOIN acme_catalog.gold.agg_sales_by_month ly 
  ON cy.year = ly.year + 1 
  AND cy.month = ly.month
  AND cy.division_id = ly.division_id
  AND cy.category_id = ly.category_id;


  -- Primer: Top 10 YoY revenue growth segments u 2011
SELECT 
  year, month, month_name,
  division_id, category_id,
  current_revenue,
  last_year_revenue,
  yoy_revenue_change,
  yoy_revenue_pct
FROM acme_catalog.gold.agg_yoy_comparison
WHERE year = 2011
ORDER BY yoy_revenue_pct DESC
LIMIT 10;

-- Primer: Segmenti sa najvećim padom u 2012
SELECT 
  year, month, month_name,
  division_id, category_id,
  current_revenue,
  last_year_revenue,
  yoy_revenue_change,
  yoy_revenue_pct
FROM acme_catalog.gold.agg_yoy_comparison
WHERE year = 2012 AND yoy_revenue_pct < 0
ORDER BY yoy_revenue_pct ASC
LIMIT 10;

-------------------------------------------------

CREATE OR REPLACE TABLE acme_catalog.gold.agg_customer_lifetime_metrics AS
WITH customer_orders AS (
  SELECT 
    fo.customer_sk,
    dc.company_name,
    dc.division_id,
    dc.division_name,
    dc.country,
    dd.full_date as order_date,
    dd.year as order_year,
    dd.month as order_month,
    fo.order_id,
    SUM(fol.revenue) as order_revenue,
    SUM(fol.margin) as order_margin
  FROM acme_catalog.gold.fact_orders fo
  JOIN acme_catalog.gold.fact_order_lines fol ON fo.order_id = fol.order_id
  JOIN acme_catalog.gold.dim_customers dc ON fo.customer_sk = dc.customer_sk AND dc.is_current = TRUE
  JOIN acme_catalog.gold.dim_date dd ON fo.date_id = dd.date_id
  GROUP BY fo.customer_sk, dc.company_name, dc.division_id, dc.division_name, dc.country,
           dd.full_date, dd.year, dd.month, fo.order_id
),
customer_summary AS (
  SELECT 
    customer_sk,
    company_name,
    division_id,
    division_name,
    country,
    -- First and last order dates
    MIN(order_date) as first_order_date,
    MAX(order_date) as last_order_date,
    -- Order metrics
    COUNT(DISTINCT order_id) as total_orders,
    COUNT(DISTINCT order_year) as years_active,
    COUNT(DISTINCT CONCAT(order_year, '-', LPAD(order_month, 2, '0'))) as months_active,
    -- Revenue metrics
    SUM(order_revenue) as lifetime_revenue,
    SUM(order_margin) as lifetime_margin,
    AVG(order_revenue) as avg_order_value,
    MIN(order_revenue) as min_order_value,
    MAX(order_revenue) as max_order_value
  FROM customer_orders
  GROUP BY customer_sk, company_name, division_id, division_name, country
)
SELECT 
  customer_sk,
  company_name,
  division_id,
  division_name,
  country,
  -- Order dates
  first_order_date,
  last_order_date,
  DATEDIFF(CURRENT_DATE(), last_order_date) as days_since_last_order,
  DATEDIFF(last_order_date, first_order_date) as customer_tenure_days,
  -- Order metrics
  total_orders,
  years_active,
  months_active,
  -- Revenue metrics
  CAST(lifetime_revenue AS DECIMAL(15,2)) as lifetime_revenue,
  CAST(lifetime_margin AS DECIMAL(15,2)) as lifetime_margin,
  ROUND((lifetime_margin / NULLIF(lifetime_revenue, 0)) * 100, 2) as lifetime_margin_pct,
  CAST(avg_order_value AS DECIMAL(12,2)) as avg_order_value,
  CAST(min_order_value AS DECIMAL(12,2)) as min_order_value,
  CAST(max_order_value AS DECIMAL(12,2)) as max_order_value,
  -- Customer value per day
  ROUND(lifetime_revenue / NULLIF(DATEDIFF(last_order_date, first_order_date), 0), 2) as avg_revenue_per_day,
  -- Customer segmentation
  CASE 
    WHEN DATEDIFF(CURRENT_DATE(), last_order_date) <= 90 THEN 'Active'
    WHEN DATEDIFF(CURRENT_DATE(), last_order_date) <= 180 THEN 'At Risk'
    WHEN DATEDIFF(CURRENT_DATE(), last_order_date) <= 365 THEN 'Dormant'
    ELSE 'Churned'
  END as customer_status,
  CASE 
    WHEN lifetime_revenue >= 100000 THEN 'VIP'
    WHEN lifetime_revenue >= 50000 THEN 'High Value'
    WHEN lifetime_revenue >= 20000 THEN 'Medium Value'
    ELSE 'Low Value'
  END as customer_tier
FROM customer_summary;

-- Primer 1: Top 10 customers po lifetime revenue
SELECT 
  company_name,
  country,
  lifetime_revenue,
  lifetime_margin,
  total_orders,
  avg_order_value,
  customer_status,
  customer_tier
FROM acme_catalog.gold.agg_customer_lifetime_metrics
ORDER BY lifetime_revenue DESC
LIMIT 10;

-- Primer 2: Customer retention analysis
SELECT 
  customer_status,
  COUNT(*) as customer_count,
  SUM(lifetime_revenue) as total_revenue,
  AVG(lifetime_revenue) as avg_lifetime_revenue,
  AVG(days_since_last_order) as avg_days_since_last_order
FROM acme_catalog.gold.agg_customer_lifetime_metrics
GROUP BY customer_status
ORDER BY 
  CASE customer_status
    WHEN 'Active' THEN 1
    WHEN 'At Risk' THEN 2
    WHEN 'Dormant' THEN 3
    WHEN 'Churned' THEN 4
  END;

-- Primer 3: Customer tier distribution
SELECT 
  customer_tier,
  COUNT(*) as customer_count,
  SUM(lifetime_revenue) as total_revenue,
  AVG(total_orders) as avg_orders,
  AVG(avg_order_value) as avg_order_value
FROM acme_catalog.gold.agg_customer_lifetime_metrics
GROUP BY customer_tier
ORDER BY 
  CASE customer_tier
    WHEN 'VIP' THEN 1
    WHEN 'High Value' THEN 2
    WHEN 'Medium Value' THEN 3
    WHEN 'Low Value' THEN 4
  END;

-- Primer 4: At-risk customers (za retention campaign)
SELECT 
  company_name,
  country,
  lifetime_revenue,
  total_orders,
  last_order_date,
  days_since_last_order
FROM acme_catalog.gold.agg_customer_lifetime_metrics
WHERE customer_status = 'At Risk'
  AND customer_tier IN ('VIP', 'High Value')
ORDER BY lifetime_revenue DESC;